# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivorship dataset using the `mlcroissant` library. The dataset includes clinicopathological and molecular variables, with detailed schema using Croissant, and is particularly suitable for secondary analysis, validation studies, and clinical data modeling.

### Dataset Source
The dataset is described and accessed via a Croissant schema at the following URL:
**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

In [ ]:
# Ensure mlcroissant is installed. If running on Colab or a new environment, uncomment below:
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Number of authors: {len(metadata.author)}")

## 2. Data Overview
List available record sets and their field/column IDs in the dataset schema. The `@id` of each entity is used as reference.

**Note**: The primary tabular data is usually in the main record set. Let's inspect all record sets and their fields using their `@id`.

In [ ]:
# List record sets and their field ids
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata. Attempting to infer record set IDs from data...")
    # Try to extract from dataset.records() generators (mlcroissant auto-discovers)
    discovered = list(dataset.record_set_ids)
    print(f"Discovered record sets: {discovered}")
    record_sets = discovered
else:
    # If properly declared in schema (as Croissant objects)
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs) for rs in record_sets]
    print(f"Found record set IDs from metadata: {record_sets}")

# For each record set, print its fields/columns
for rs_id in record_sets:
    print(f"\nRecord Set @id: {rs_id}")
    try:
        fields = dataset.record_set_fields(rs_id)
        field_ids = [field['@id'] for field in fields]
        print(f"Fields: {field_ids}")
    except Exception as e:
        print(f"Could not fetch fields for record set {rs_id}: {e}")

## 3. Data Extraction
Extract data from a record set into a pandas DataFrame for further analysis. Use the record set and field `@id`s from the overview above.

Let's demonstrate extracting tabular data from each discovered record set.

In [ ]:
all_record_sets = record_sets
dataframes = {}

for rs_id in all_record_sets:
    print(f"\nLoading data for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows. Columns (@id): {list(df.columns)}")
        else:
            print("No records found for this record set.")
    except Exception as e:
        print(f"Error loading data for {rs_id}: {e}")

# Choose the main tabular record set (typically the one with most fields/columns or largest size)
if len(dataframes) > 0:
    # Heuristic: take the dataframe with most columns
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
    print(f"\nMain record set selected for analysis: {main_rs_id}")
    print("Sample columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No tabular dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Now let's analyze key variables from the dataset. We'll filter, normalize, and group data using field `@id`s only.

*First, select a numeric field for analysis. For demonstration, we'll try to infer an appropriate numeric @id from the column list (e.g., age or interval columns, or any integer).*

In [ ]:
# Main record set id and dataframe
record_set_id = main_rs_id
df = dataframes[record_set_id]

# Try to suggest a numeric field by @id (e.g., containing 'age', 'interval', or numeric dtype)
possible_numeric = [col for col in df.columns if any(x in col.lower() for x in ['age','interval','years','metastasis','count'])]
numeric_field_id = None
for col in possible_numeric:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: use the first int/float column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Selected numeric field for EDA: {numeric_field_id}")
if numeric_field_id is None:
    print("No suitable numeric field detected; please check the schema or field list.")
else:
    # Threshold: 10 (as in the template), or use median/quantile for demo
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Head of normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # For grouping, suggest a likely categorical field by @id
    # For medical data, might be 'sex', 'msi_status', 'anatomical_location', etc.
    cat_guess = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'msi','site', 'anatom', 'location','type','group'])]
    group_field_id = cat_guess[0] if cat_guess else None
    print(f"\nAttempted group-by field: {group_field_id}")
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and its group-wise statistics if possible.

We will use matplotlib and seaborn for plots (install if needed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by a grouping field
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook we demonstrated:
- Loading a Croissant metadata schema and extracting tabular data using only `@id`s
- Listing record sets, fields, and exploring their structure
- Performing basic EDA: filtering, normalization, and group-wise summary
- Plotting distributions and group statistics

The FAIR² dataset is rich in clinicopathological variables about second primary colorectal cancer in survivors, and suitable for biomarker stratification, model development, and reproducible clinical research pipelines using Croissant and `mlcroissant`.

You can extend this workflow to additional analytic tasks, always referencing fields by their `@id` as per Croissant best practices.